In [2]:
import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import torch
from tqdm import tqdm


EMBED_DIM = 128
HIDDEN_DIM = 64

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"))

Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.


In [3]:
df["clean_text"] = df["Text"].str.lower()
df["clean_text"] = df["clean_text"].str.replace(r"<[^>]+>", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"[^\w\s]", " ", regex=True)

In [4]:
def tokenize(text, vocab, maxlen=200):
    tokens = text.split()[:maxlen]
    ids = [vocab.get(t, 1) for t in tokens]
    ids += [0] * (maxlen - len(ids))
    return ids


In [5]:
from collections import Counter

# 1. Construire le vocabulaire
all_words = " ".join(df["clean_text"]).split()
vocab = {"<PAD>": 0, "<UNK>": 1}
# 22 000 mots sont présents 20 fois ou plus dans les reviews, on ignore les autres
for word, count in Counter(all_words).most_common(20000):
    vocab[word] = len(vocab)

vocab_size = len(vocab)
# 95% des reviews ont moins de 222 tokens

df["tokenized"] = df["clean_text"].apply(tokenize, vocab=vocab, maxlen=222)

In [6]:
# Sampling dataset
# Train = 70% / Validation = 15% / Test = 15%
train_df = df.sample(frac=0.7, random_state=42)
#train_df = df.sample(n=100, random_state=42)
temp_df = df.drop(train_df.index)
#val_df = temp_df.sample(n=20, random_state=42)
val_df = temp_df.sample(frac=0.15, random_state=42)
temp_df = temp_df.drop(val_df.index)
test_df = temp_df.sample(frac=0.15, random_state=42)

In [7]:
class ReviewDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.data = df[["tokenized", "Score"]].values
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, score = self.data[idx]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(score -1, dtype=torch.long)

In [8]:
class ReviewModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, bidirectional= True, batch_first= True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(out)

In [9]:
dataloader_train = DataLoader(ReviewDataset(train_df), batch_size=32, shuffle=True)
dataloader_val = DataLoader(ReviewDataset(val_df), batch_size=32)
dataloader_test = DataLoader(ReviewDataset(test_df), batch_size=32)

device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

model = ReviewModel(vocab_size, embed_dim=128, hidden_dim=64, num_classes=len(df['Score'].unique())).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
MAX_EPOCHS = 3
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for X_batch, Y_batch in tqdm(dataloader_train, desc=f"Epoch {epoch+1} - Training"):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()
        predicted = model(X_batch)
        loss = criterion(predicted, Y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(dataloader_val, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            predicted = model(X_batch)
            predicted_value = predicted.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += criterion(predicted, Y_batch).item()
    val_accuracy = correct / total
    val_loss /= len(dataloader_val)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f}")
    wandb.log({"val_accuracy": val_accuracy, "val_loss": val_loss})


In [ ]:
torch.save(model.state_dict(), "../model.pt")